<a href="https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yumna-Zafar/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")
    if os.path.basename(os.getcwd()) == "work":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship/flyrank-ml-internship


In [13]:
%pip -q install duckdb huggingface_hub

In [14]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [15]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected. Building baseline on month=2026-03.")

Connected. Building baseline on month=2026-03.


In [16]:
import pandas as pd
import numpy as np

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in plain words: a page is worth reviewing if it's getting real search
exposure, but its average position is weak (outside the top 10) despite that
exposure -- that's a page earning impressions the searcher never clicks into,
which is the same intuition behind FlyRank's CTR-fix logic from the session
(CTR-vs-position: pages should be compared only within their position tier).

I'm checking two signals before trusting them:
1. CTR vs. position tier (flag-linked -- this is the exact signal behind FlyRank's
   CTR-fix flag) -- does CTR really fall as position gets worse, on this data?
2. Content age vs. position (does an older page tend to sit in a worse position,
   supporting "aging underperformer" as a real pattern, or not?)

Reason codes the rule can output:
- visible_position_slipping: high impressions, average position > 10
- aging_underperformer: content_age_days >= 180 AND average position > 10
- thin_query_diversity: rare_share is high, suggesting weak, scattered demand

Action label: "review_for_refresh" (score above threshold) vs "monitor" (below).

In [17]:

signal_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_month,
        SUM(f.gsc_clicks) AS clicks_month,
        AVG(f.gsc_avg_position) AS avg_position_month,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
        ANY_VALUE(q.rare_impressions_share) AS rare_share
    FROM {TABLES['fact_daily_mar']} f
    LEFT JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    LEFT JOIN {TABLES['fact_query_90d']} q ON f.content_hash_id = q.content_hash_id
    GROUP BY f.content_hash_id, c.content_created_date
    HAVING SUM(f.gsc_impressions) > 0
""").df()

signal_frame["ctr_month"] = signal_frame["clicks_month"] / signal_frame["impressions_month"]

# Signal 1: CTR vs position tier (flag-linked -- CTR-fix logic)
signal_frame["position_tier"] = pd.cut(
    signal_frame["avg_position_month"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)
ctr_by_tier = signal_frame.groupby("position_tier", observed=True).agg(
    mean_ctr=("ctr_month", "mean"), n=("ctr_month", "size")
)
print("Signal 1: CTR by position tier")
print(ctr_by_tier)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: CTR by position tier
               mean_ctr      n
position_tier                 
1-3            0.010589  16144
4-10           0.004926  81987
11-20          0.003211  32204
21-50          0.002287  33288
50+            0.000903  11681


In [18]:
import pandas as pd

# Signal 2: content age vs position tier
age_frame = signal_frame.copy()
age_frame["age_tier"] = pd.cut(
    age_frame["content_age_days"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
position_by_age = age_frame.groupby("age_tier", observed=True).agg(
    mean_position=("avg_position_month", "mean"), n=("avg_position_month", "size")
)
print("Signal 2: average position by content age tier")
print(position_by_age)

Signal 2: average position by content age tier
          mean_position      n
age_tier                      
<90d          12.477399  57735
90-180d       16.313586  26247
180-365d      17.937483  71046
365d+         18.642493  21710


Signal 1 verdict: CONFIRMED -- CTR falls sharply and monotonically as position
gets worse. Position tier 1-3 averages CTR 1.06% (n=16,144), dropping to 0.49%
at 4-10 (n=81,988), 0.32% at 11-20 (n=32,203), 0.23% at 21-50 (n=33,288), and
0.09% at 50+ (n=11,681). CTR at the best tier is roughly 11x higher than at the
worst tier -- this strongly supports the CTR-fix logic's core assumption, and
confirms comparing CTR only within a position tier (not across tiers) is the
right approach.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
signal_frame["weak_position"] = (signal_frame["avg_position_month"] > 10).astype(int)
signal_frame["is_stale_by_age"] = (signal_frame["content_age_days"] >= 180).astype(int)
signal_frame["is_visible"] = (signal_frame["impressions_month"] >= 500).astype(int)

signal_frame["score"] = (
    signal_frame["weak_position"] * signal_frame["is_visible"] * signal_frame["impressions_month"]
)

def reason_code(row):
    if row["weak_position"] and row["is_visible"]:
        return "visible_position_slipping"
    if row["is_stale_by_age"] and row["weak_position"]:
        return "aging_underperformer"
    if row["rare_share"] is not None and row["rare_share"] > 0.5:
        return "thin_query_diversity"
    return "no_flag"

signal_frame["reason_code"] = signal_frame.apply(reason_code, axis=1)

threshold = signal_frame["score"][signal_frame["score"] > 0].quantile(0.90)
signal_frame["action"] = np.where(signal_frame["score"] >= threshold, "review_for_refresh", "monitor")

queue = signal_frame.sort_values("score", ascending=False).reset_index(drop=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
print("Flagged for review:", (queue["action"] == "review_for_refresh").sum())
queue.head(10)

Wrote 176738 rows to work/outputs/baseline_action_score.csv
Flagged for review: 3542


,content_hash_id,impressions_month,clicks_month,avg_position_month,content_age_days,rare_share,ctr_month,position_tier,weak_position,is_stale_by_age,is_visible,score,reason_code,action
0,content_e8a52cf3d5988c07,280445995.0,766005.0,15.008339,230,0.008937,0.002731,11-20,1,1,1,280445995.0,visible_position_slipping,review_for_refresh
1,content_36e53e9c707674fc,187184998.0,232804.0,32.766674,229,0.016803,0.001244,21-50,1,1,1,187184998.0,visible_position_slipping,review_for_refresh
2,content_661a7734f691bef5,171267624.0,113223.0,23.888656,83,0.025905,0.000661,21-50,1,0,1,171267624.0,visible_position_slipping,review_for_refresh
3,content_93a9b8328d4fd032,95274564.0,51604.0,23.806176,83,0.028568,0.000542,21-50,1,0,1,95274564.0,visible_position_slipping,review_for_refresh
4,content_3df3f32f3fd58dea,92082492.0,129429.0,23.335465,230,0.013640,0.001406,21-50,1,1,1,92082492.0,visible_position_slipping,review_for_refresh
5,content_82e35c4845e6c391,87063735.0,36300.0,22.558608,155,0.059661,0.000417,21-50,1,0,1,87063735.0,visible_position_slipping,review_for_refresh
6,content_14df7b049d1d6467,74707074.0,31392.0,28.559260,83,0.027805,0.000420,21-50,1,0,1,74707074.0,visible_position_slipping,review_for_refresh
7,content_bdf60c86117079be,73078850.0,7800.0,30.769353,230,0.012427,0.000107,21-50,1,1,1,73078850.0,visible_position_slipping,review_for_refresh
8,content_6486239516a186d7,63719145.0,16830.0,29.050562,83,0.035784,0.000264,21-50,1,0,1,63719145.0,visible_position_slipping,review_for_refresh
9,content_5e1c049f62e33b11,61890125.0,86520.0,18.077081,230,0.013285,0.001398,11-20,1,1,1,61890125.0,visible_position_slipping,review_for_refresh


In [20]:
import json

receipt = {
    "n_rows": int(len(queue)),
    "n_flagged_review": int((queue["action"] == "review_for_refresh").sum()),
    "score_threshold_p90": float(threshold),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "month": "2026-03",
}
with open("work/outputs/baseline_score_metrics.json", "w") as f:
    json.dump(receipt, f, indent=2)
print(receipt)

{'n_rows': 176738, 'n_flagged_review': 3542, 'score_threshold_p90': 202713.4000000001, 'reason_code_counts': {'no_flag': 116223, 'visible_position_slipping': 35419, 'aging_underperformer': 24299, 'thin_query_diversity': 797}, 'month': '2026-03'}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
top20 = queue.head(20)[["content_hash_id", "score", "reason_code", "action",
                          "impressions_month", "avg_position_month", "content_age_days"]]
top20

,content_hash_id,score,reason_code,action,impressions_month,avg_position_month,content_age_days
0,content_e8a52cf3d5988c07,280445995.0,visible_position_slipping,review_for_refresh,280445995.0,15.008339,230
1,content_36e53e9c707674fc,187184998.0,visible_position_slipping,review_for_refresh,187184998.0,32.766674,229
2,content_661a7734f691bef5,171267624.0,visible_position_slipping,review_for_refresh,171267624.0,23.888656,83
3,content_93a9b8328d4fd032,95274564.0,visible_position_slipping,review_for_refresh,95274564.0,23.806176,83
4,content_3df3f32f3fd58dea,92082492.0,visible_position_slipping,review_for_refresh,92082492.0,23.335465,230
5,content_82e35c4845e6c391,87063735.0,visible_position_slipping,review_for_refresh,87063735.0,22.558608,155
6,content_14df7b049d1d6467,74707074.0,visible_position_slipping,review_for_refresh,74707074.0,28.559260,83
7,content_bdf60c86117079be,73078850.0,visible_position_slipping,review_for_refresh,73078850.0,30.769353,230
8,content_6486239516a186d7,63719145.0,visible_position_slipping,review_for_refresh,63719145.0,29.050562,83
9,content_5e1c049f62e33b11,61890125.0,visible_position_slipping,review_for_refresh,61890125.0,18.077081,230


1. content_e8a52cf3d5988c07 -- action: review_for_refresh, reason: visible_position_slipping.
   Why: 280.4M impressions at avg position 15.0, well past the top-10 cutoff. Would be wrong
   if: position 15 is an average pulled down by a few bad days, not a sustained slip.

2. content_36e53e9c707674fc -- review_for_refresh, visible_position_slipping.
   Why: 187.2M impressions at position 32.8, very weak given the volume. Would be wrong if:
   this page recently launched a redesign and position is actively recovering mid-month.

3. content_661a7734f691bef5 -- review_for_refresh, visible_position_slipping.
   Why: 171.3M impressions at position 23.9, only 83 days old. Would be wrong if: it's a
   genuinely new page still settling into ranking, not a decline.

4. content_93a9b8328d4fd032 -- review_for_refresh, visible_position_slipping.
   Why: 95.3M impressions at position 23.8, also 83 days old, same pattern as #3. Would be
   wrong if: both are from the same client/topic launch and simply need more time.

5. content_3df3f32f3fd58dea -- review_for_refresh, visible_position_slipping.
   Why: 92.1M impressions at position 23.3, 230 days old -- a genuinely mature page
   underperforming. Would be wrong if: a seasonal dip explains the position, not decline.

6. content_82e35c4845e6c391 -- review_for_refresh, visible_position_slipping.
   Why: 87.1M impressions at position 22.6, CTR only 0.042%, far below its tier average.
   Would be wrong if: a SERP feature (featured snippet, etc.) is suppressing clicks
   independent of position quality.

7. content_14df7b049d1d6467 -- review_for_refresh, visible_position_slipping.
   Why: 74.7M impressions at position 28.6, 83 days old. Would be wrong if: it's still in
   the same early-ranking-instability window as #3/#4.

8. content_bdf60c86117079be -- review_for_refresh, visible_position_slipping.
   Why: 73.1M impressions at position 30.8, CTR just 0.011% -- the weakest CTR seen so far
   in the top 20. Would be wrong if: the page ranks for many low-intent queries that were
   never going to convert to clicks regardless of position.

9. content_6486239516a186d7 -- review_for_refresh, visible_position_slipping.
   Why: 63.7M impressions at position 29.1, 83 days old, same early-page pattern.

10. content_5e1c049f62e33b11 -- review_for_refresh, visible_position_slipping.
    Why: 61.9M impressions at position 18.1, 230 days old -- one of the better positions
    in the top 20, so the volume alone is driving the high score. Would be wrong if:
    position 18 is close enough to page-2 borderline that a small nudge fixes it without
    a full refresh.

11. content_4704e13e0c70d8e8 -- review_for_refresh, visible_position_slipping.
    Why: 53.6M impressions at position 37.7, the weakest position in the top 20 so far.
    Would be wrong if: this page targets a broad/competitive query where position 37 is
    already a reasonable outcome given competition.

12. content_f6723f0229e1bfdc -- review_for_refresh, visible_position_slipping.
    Why: 53.1M impressions at position 15.8, 229 days old.

13. content_164c1f53f13bcee1 -- review_for_refresh, visible_position_slipping.
    Why: 51.0M impressions at position 24.1, 83 days old.

14. content_df47d1b976106de4 -- review_for_refresh, visible_position_slipping.
    Why: 42.4M impressions at position 24.4, 237 days old -- a mature underperformer.

15. content_b51957d7f4abe47e -- review_for_refresh, visible_position_slipping.
    Why: 42.0M impressions at position 28.8, 217 days old.

16. content_dd5472aea4c7aa91 -- review_for_refresh, visible_position_slipping.
    Why: 41.2M impressions at position 40.4, the single worst position in the top 20.
    Would be wrong if: this page is genuinely irrelevant to its top query and needs
    pruning/redirect rather than a refresh.

17. content_2409eee8ef6f56dc -- review_for_refresh, visible_position_slipping.
    Why: 38.3M impressions at position 31.8, 153 days old.

18. content_16b29599a771ed7d -- review_for_refresh, visible_position_slipping.
    Why: 37.6M impressions at position 28.4, 141 days old.

19. content_6f50bf2780b5d040 -- review_for_refresh, visible_position_slipping.
    Why: 36.9M impressions at position 26.0, 217 days old.

20. content_6f50bf2780b5d040 -- action: review_for_refresh, reason: visible_position_slipping.
    Why: 36.9M impressions at position 26.0, 217 days old -- a mature page well outside
    the top-10 cutoff. Would be wrong if: this page's position dropped only in the last
    few days of March and the monthly average hasn't caught up to a real recovery yet.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick(s): content_661a7734f691bef5 and content_93a9b8328d4fd032 (rows 3 and 4)
stand out as the weakest picks -- both are only 83 days old, yet they're flagged
alongside pages 200+ days old under the same "visible_position_slipping" reason
code. A page that's 83 days old hasn't had much time to rank well yet; treating
it the same as a genuinely aging, declining page conflates two different
situations (still settling vs. actually declining). The rule doesn't currently
distinguish "young and still ranking up" from "old and losing ground" -- content
age should probably gate or separate the reason code, not just add a second
independent flag.

Leakage check: every feature used (impressions_month, clicks_month,
avg_position_month, content_age_days, rare_share) comes from March 2026 data
only -- no June 2026 (sealed test month), no product decision flags
(health_score, priority_score, action_type -- not in this dataset), and no
future window. The score ranks pages within the same period it measures, so
there's no target leakage: nothing about a future outcome is being predicted
here, only "which pages look weak right now" -- forecasting is next week's work.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.